# <span style="color: #1F1DB5;">SPRINT 17 – NLP y Textos

# <span style="color: #1F1DB5;">Notebook 1 – Introducción al Procesamiento de Lenguaje Natural (NLP) — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender el pipeline completo de NLP: tokenización, limpieza, vectorización, modelado.
- Aplicar técnicas de preprocesamiento de texto: stopwords, stemming, lemmatization.
- Representar texto numéricamente con Bag of Words (CountVectorizer) y TF-IDF.
- Entrenar un clasificador de texto con LogisticRegression para análisis de sentimientos.
- Evaluar modelos de clasificación de texto con métricas apropiadas.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender el **pipeline completo de NLP**: tokenización, limpieza, vectorización, modelado.
- Aplicar técnicas de **preprocesamiento de texto**: stopwords, stemming, lemmatization.
- Representar texto numéricamente con **Bag of Words** (CountVectorizer) y **TF-IDF**.
- Entrenar un **clasificador de texto** con LogisticRegression para análisis de sentimientos.
- Evaluar modelos de clasificación de texto con métricas apropiadas.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Pipeline de NLP y preprocesamiento | 25 min |
Bag of Words y TF-IDF | 25 min |
Clasificación de texto con sklearn | 25 min |
Presentaciones de Estudiantes | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo puede una máquina entender el lenguaje humano?

Cada vez que Netflix te recomienda una película por su descripción, o Gmail detecta spam, o un chatbot te atiende, hay NLP detrás. El reto fundamental es: las computadoras solo entienden números, pero el lenguaje humano es texto. ¿Cómo cerramos esa brecha?

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Pipeline de NLP y Preprocesamiento (25 mins)

El **pipeline de NLP** transforma texto crudo en algo que un modelo de ML pueda usar:

```
Texto crudo → Limpieza → Tokenización → Normalización → Vectorización → Modelo
```

**1. Limpieza:**
- Convertir a minúsculas.
- Eliminar puntuación, números (si no aportan), caracteres especiales.
- Eliminar HTML, URLs, emojis (según el caso).

**2. Tokenización:**
- Dividir el texto en unidades (tokens): palabras, oraciones o subpalabras.
- "El gato come pescado" → ["el", "gato", "come", "pescado"]

**3. Stopwords:**
- Palabras muy frecuentes que no aportan significado: "el", "de", "que", "y", "en"...
- Se eliminan porque agregan ruido sin información.

**4. Stemming vs Lemmatization:**
- **Stemming:** Corta sufijos para obtener la raíz. "corriendo" → "corr". Rápido pero agresivo.
- **Lemmatization:** Busca la forma base real. "corriendo" → "correr". Más preciso pero lento.

**Analogía:** Es como preparar ingredientes antes de cocinar. No metes la bolsa del supermercado directo a la olla; lavas, pelas, cortas. Igual con el texto.

Veamos cómo se implementa cada paso del pipeline con NLTK y expresiones regulares. Prepararemos un mini corpus de reseñas de películas.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Descarga recursos necesarios (solo la primera vez)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Corpus de ejemplo: reseñas de películas
reviews = [
    "¡Esta película es INCREÍBLE! Los efectos especiales son de otro mundo. 10/10",
    "Aburrida y predecible. No la recomiendo para nada, perdí mi tiempo.",
    "Buena actuación del protagonista, pero la historia es muy lenta.",
    "La mejor película que he visto este año. Lloré de emoción.",
    "Terrible. El guión no tiene sentido y las actuaciones son pésimas."
]

# Paso 1: Limpieza
def limpiar_texto(texto):
    texto = texto.lower()                          # Minúsculas
    texto = re.sub(r'[^a-záéíóúñü\s]', '', texto)  # Solo letras y espacios
    texto = re.sub(r'\s+', ' ', texto).strip()      # Espacios múltiples
    return texto

# Paso 2: Tokenización
def tokenizar(texto):
    return word_tokenize(texto, language='spanish')

# Paso 3: Eliminar stopwords
stop_es = set(stopwords.words('spanish'))
def quitar_stopwords(tokens):
    return [t for t in tokens if t not in stop_es]

# Paso 4: Stemming
stemmer = SnowballStemmer('spanish')
def aplicar_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

# Demostración paso a paso
texto = reviews[0]
print(f"Original: {texto}")
print(f"Limpio:   {limpiar_texto(texto)}")
tokens = tokenizar(limpiar_texto(texto))
print(f"Tokens:   {tokens}")
sin_stop = quitar_stopwords(tokens)
print(f"Sin stop: {sin_stop}")
stems = aplicar_stemming(sin_stop)
print(f"Stems:    {stems}")

Preguntas:

- ¿Por qué eliminamos las stopwords? ¿Hay casos donde sería mejor conservarlas?

- ¿Cuál es la diferencia práctica entre stemming y lemmatization? ¿Cuándo usarías cada uno?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Bag of Words y TF-IDF (25 mins)

Ya sabemos limpiar texto. Ahora necesitamos convertirlo en **números** que un modelo entienda. Las dos técnicas clásicas son:

**Bag of Words (BoW) – CountVectorizer:**
- Crea un vocabulario con todas las palabras únicas.
- Cada documento se representa como un vector de conteos.
- "el gato come" y "come el perro" → vocabulario: [come, el, gato, perro] → [1,1,1,0] y [1,1,0,1]
- **Problema:** No distingue palabras importantes de frecuentes. "El" aparece mucho pero no dice nada.

**TF-IDF (Term Frequency – Inverse Document Frequency):**
- **TF:** Frecuencia del término en el documento (cuánto aparece en ESTE documento).
- **IDF:** Inverso de la frecuencia en el corpus (cuán raro es en TODOS los documentos).
- TF-IDF = TF × IDF → Palabras frecuentes en un doc pero raras en el corpus tienen score alto.
- Resuelve el problema de BoW: penaliza palabras muy comunes y destaca las discriminantes.

**Ambos producen matrices sparse** (la mayoría de valores son 0, porque cada documento usa pocas palabras del vocabulario total).

Implementemos ambas técnicas y comparemos cómo representan nuestras reseñas.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Preprocesar todas las reseñas
def preprocesar(texto):
    texto = limpiar_texto(texto)
    tokens = tokenizar(texto)
    tokens = quitar_stopwords(tokens)
    return ' '.join(tokens)

reviews_clean = [preprocesar(r) for r in reviews]
print("Reseñas preprocesadas:")
for i, r in enumerate(reviews_clean):
    print(f"  {i+1}. {r}")

# Bag of Words
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(reviews_clean)

print(f"\n--- Bag of Words ---")
print(f"Vocabulario ({len(count_vec.get_feature_names_out())} palabras): {count_vec.get_feature_names_out()[:15]}...")
print(f"Matriz shape: {bow_matrix.shape}")
print(f"Tipo: {type(bow_matrix)} (sparse!)")

# TF-IDF
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(reviews_clean)

print(f"\n--- TF-IDF ---")
print(f"Matriz shape: {tfidf_matrix.shape}")

# Comparación para la primera reseña
import pandas as pd
vocab = count_vec.get_feature_names_out()
comparacion = pd.DataFrame({
    'Palabra': vocab,
    'BoW (conteo)': bow_matrix[0].toarray().flatten(),
    'TF-IDF (peso)': tfidf_matrix[0].toarray().flatten().round(3)
})
# Solo palabras con valor > 0
print("\nReseña 1 - Comparación BoW vs TF-IDF:")
print(comparacion[comparacion['BoW (conteo)'] > 0].to_string(index=False))

Preguntas:

- ¿Por qué TF-IDF da scores diferentes a palabras que tienen el mismo conteo en BoW?

- ¿Qué tipo de palabras tendrán TF-IDF alto? ¿Y cuáles tendrán TF-IDF cercano a 0?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Clasificación de Texto con sklearn (25 mins)

Ahora unimos todo: preprocesamiento + vectorización + modelo de clasificación. Vamos a construir un **clasificador de sentimientos** que determine si una reseña es positiva o negativa.

El flujo completo es:
1. Dataset con textos etiquetados (positivo/negativo).
2. Preprocesamiento (limpieza, tokenización).
3. Vectorización (TF-IDF funciona mejor que BoW para clasificación).
4. Modelo (LogisticRegression es excelente para texto: rápido, interpretable).
5. Evaluación (precision, recall, F1-score).

**¿Por qué LogisticRegression para texto?**
- Funciona muy bien con features de alta dimensión (muchas palabras).
- Es rápido de entrenar.
- Los coeficientes te dicen qué palabras son más indicativas de cada clase.

Creemos un dataset más grande de reseñas y entrenemos nuestro clasificador completo.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# Dataset de reseñas (simulado pero realista)
np.random.seed(42)
positive_reviews = [
    "Excelente película, la mejor del año sin duda",
    "Me encantó la historia, muy emotiva y bien actuada",
    "Increíble fotografía y banda sonora espectacular",
    "Una obra maestra del cine contemporáneo",
    "Muy divertida, me reí durante toda la función",
    "El director se lució con esta producción",
    "Actuaciones memorables de todo el elenco",
    "La recomiendo totalmente, vale cada peso",
    "Una historia que te atrapa desde el primer minuto",
    "Perfecta combinación de acción y drama",
    "Salí del cine emocionado, película brillante",
    "El mejor guión que he visto en mucho tiempo",
    "Fantástica, la vi dos veces y me gustó más la segunda",
    "Muy buena trama y excelentes efectos visuales",
    "Me hizo llorar de alegría, hermosa película",
]
negative_reviews = [
    "Pésima película, no la recomiendo a nadie",
    "Aburrida y sin sentido, perdí dos horas de mi vida",
    "Actuaciones terribles y un guión predecible",
    "La peor película que he visto este año",
    "No entiendo las buenas críticas, es muy mala",
    "Me dormí a la mitad, extremadamente lenta",
    "Efectos especiales baratos y historia vacía",
    "Decepcionante, esperaba mucho más del director",
    "No vale la pena, ahórrate el dinero del boleto",
    "Horrible, no tiene ni pies ni cabeza",
    "El final es ridículo y arruina toda la película",
    "Muy mala, las actuaciones son de telenovela",
    "Insulta la inteligencia del espectador",
    "Repetitiva y sin ninguna originalidad",
    "La peor inversión de tiempo posible",
]

texts = positive_reviews + negative_reviews
labels = [1]*len(positive_reviews) + [0]*len(negative_reviews)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

# Pipeline completo: TF-IDF + LogisticRegression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("=== Clasificador de Sentimientos ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Negativa', 'Positiva'])}")

# Palabras más indicativas
feature_names = pipeline['tfidf'].get_feature_names_out()
coefs = pipeline['clf'].coef_[0]
top_pos = np.argsort(coefs)[-5:]
top_neg = np.argsort(coefs)[:5]
print("Palabras más POSITIVAS:", [feature_names[i] for i in top_pos])
print("Palabras más NEGATIVAS:", [feature_names[i] for i in top_neg])

Preguntas:

- ¿Qué palabras son más indicativas de sentimiento positivo y cuáles de negativo?

- ¿Por qué usamos un Pipeline en vez de hacer vectorización y modelo por separado?

- ¿Cómo podrías mejorar la precisión del modelo con más datos o mejor preprocesamiento?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Presentaciones de Estudiantes (15 mins)

**Dinámica:** Grupos de 3-4 personas. Cada grupo investiga brevemente uno de estos temas y presenta en 3 minutos:

1. **Grupo 1:** ¿Qué es Named Entity Recognition (NER) y para qué sirve?
2. **Grupo 2:** ¿Cómo funciona un corrector ortográfico automático?
3. **Grupo 3:** ¿Qué es sentiment analysis y dónde se usa en la industria?
4. **Grupo 4:** ¿Qué limitaciones tiene BoW/TF-IDF para entender el lenguaje?

Pueden buscar en Google, usar sus apuntes o el notebook. Lo importante es explicar con sus propias palabras.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


## <span style="color: #2826DE;">Tips y buenas prácticas

- Siempre **explora** tu corpus antes de modelar: longitud promedio, vocabulario, distribución de clases.
- TF-IDF generalmente funciona mejor que BoW para clasificación.
- Usa `max_features` para limitar el vocabulario y evitar sobreajuste.
- El preprocesamiento depende del idioma. En español, `SnowballStemmer('spanish')` funciona bien.
- `Pipeline` de sklearn mantiene todo reproducible y evita data leakage en la vectorización.
- Para datasets pequeños, LogisticRegression suele ganar a modelos más complejos.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué es la tokenización en NLP?

- Traducir texto a otro idioma
- Dividir el texto en unidades (palabras, oraciones) 
- Eliminar palabras repetidas
- Convertir texto a números


2️⃣ ¿Qué son las stopwords?

- Palabras con errores ortográficos
- Palabras muy frecuentes que no aportan significado 
- Palabras en inglés dentro de texto en español
- Palabras con más de 10 letras


3️⃣ ¿Qué ventaja tiene TF-IDF sobre Bag of Words?

- Es más rápido de calcular
- Produce vectores más pequeños
- Penaliza palabras muy comunes y destaca las discriminantes 
- No necesita preprocesamiento


4️⃣ ¿Por qué LogisticRegression funciona bien para clasificación de texto?

- Porque maneja bien features de alta dimensión 
- Porque es un modelo no lineal
- Porque no necesita datos de entrenamiento
- Porque entiende gramática


5️⃣ ¿Qué produce CountVectorizer?

- Un resumen del texto
- Una matriz de conteos de palabras por documento 
- Una lista de sinónimos
- Un modelo de clasificación


6️⃣ ¿Cuál es el orden correcto del pipeline de NLP?

- Modelo → Vectorización → Limpieza → Tokenización
- Limpieza → Tokenización → Vectorización → Modelo 
- Vectorización → Limpieza → Modelo → Tokenización
- Tokenización → Modelo → Limpieza → Vectorización

In [ ]:
# Probemos nuestro clasificador con nuevas reseñas
nuevas_reviews = [
    "Una joya del cine, me fascinó cada minuto",
    "No vale la pena, es puro ruido sin sustancia",
    "Regular, tiene sus momentos buenos pero también malos",
    "Excelente actuación y una historia conmovedora",
    "Aburrida, predecible y con un final decepcionante"
]

print("=== Predicciones en reseñas nuevas ===\n")
for review in nuevas_reviews:
    pred = pipeline.predict([review])[0]
    proba = pipeline.predict_proba([review])[0]
    sentimiento = "POSITIVA ✅" if pred == 1 else "NEGATIVA ❌"
    confianza = max(proba)
    print(f"  '{review[:50]}...'")
    print(f"    → {sentimiento} (confianza: {confianza:.1%})\n")

Preguntas:

- ¿Qué pasa con la reseña "regular"? ¿Por qué el modelo tiene dificultades con sentimientos neutros?

- ¿Cómo podrías mejorar este clasificador sin cambiar el algoritmo?

- ¿Qué limitaciones tiene este enfoque para textos con sarcasmo o ironía?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Visualización del Vocabulario TF-IDF

Veamos cuáles son las palabras más relevantes para cada clase en nuestro modelo entrenado. Esto nos ayuda a interpretar qué está "aprendiendo" el clasificador.

In [ ]:
# Top palabras por clase
feature_names = pipeline['tfidf'].get_feature_names_out()
coefs = pipeline['clf'].coef_[0]

n_top = 10
top_positive_idx = np.argsort(coefs)[-n_top:][::-1]
top_negative_idx = np.argsort(coefs)[:n_top]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Palabras positivas
axes[0].barh(range(n_top), coefs[top_positive_idx], color='green', alpha=0.7)
axes[0].set_yticks(range(n_top))
axes[0].set_yticklabels([feature_names[i] for i in top_positive_idx])
axes[0].set_title('Top 10 palabras → POSITIVO')
axes[0].set_xlabel('Coeficiente')

# Palabras negativas
axes[1].barh(range(n_top), coefs[top_negative_idx], color='red', alpha=0.7)
axes[1].set_yticks(range(n_top))
axes[1].set_yticklabels([feature_names[i] for i in top_negative_idx])
axes[1].set_title('Top 10 palabras → NEGATIVO')
axes[1].set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

print("Estos coeficientes nos dicen qué palabras 'empujan' la predicción")
print("hacia positivo (coef > 0) o negativo (coef < 0).")